# 02 - Signal processing: preprocessing, epoching and feature extraction

This notebook demonstrates the deterministic signal-processing layer on one
recording, and then reads the extracted feature table that the pipeline saved.
It is deliberately small: the pipeline is the source of truth, and this is a
window into it.

```bash
python -m eeg_research_agent.cli features --mode smoke
```

In [ ]:
from pathlib import Path

import numpy as np

from eeg_research_agent.config import load_config
from eeg_research_agent.data.acquisition import acquire_dataset
from eeg_research_agent.data.preprocessing import preprocess_recording

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = load_config()
corpus = acquire_dataset(config, origin='synthetic')
recording = corpus.recordings[0]
print('origin  :', recording.data_origin)
print('channel :', recording.ch_name, '|', recording.sfreq, 'Hz')
print('minutes :', round(recording.signal.size / recording.sfreq / 60, 1))
print('annotations:', len(recording.annotations))

## Preprocessing

Resample to the target rate, then a zero-phase Butterworth band-pass, an optional
mains notch and detrending - all on the **continuous** signal, so filter
transients never land on an epoch boundary. The report records exactly what ran.

In [ ]:
cleaned, report = preprocess_recording(recording, config)
print(report.to_dict())

In [ ]:
from scipy import signal as sps

freq_raw, psd_raw = sps.welch(recording.signal, fs=recording.sfreq, nperseg=1024)
freq_cln, psd_cln = sps.welch(cleaned.signal, fs=cleaned.sfreq, nperseg=1024)

def band_power_uv2(freq, psd, low, high):
    mask = (freq >= low) & (freq <= high)
    return float(np.trapezoid(psd[mask], freq[mask]) * 1e12)

for name in ('delta', 'theta', 'alpha', 'sigma', 'beta'):
    band = config.features.bands[name]
    raw = band_power_uv2(freq_raw, psd_raw, *band)
    clean = band_power_uv2(freq_cln, psd_cln, *band)
    print(f'{name:<6} {band}  raw={raw:9.2f} uV^2   cleaned={clean:9.2f} uV^2')

## Epoching and the extracted feature table

Epochs are 30 s, and any exclusion is recorded with its reason rather than
silently dropped. The saved table is the input to the statistics stage.

In [ ]:
import json
import pandas as pd

saved = sorted((ROOT / 'data/features').glob('features_*.csv.gz'))
table = pd.read_csv(saved[-1])
print('feature table :', saved[-1].name)
print('shape         :', table.shape)

catalog = json.loads((ROOT / 'artifacts/tables/feature_catalog.json').read_text())
print('registered features:', catalog['n_features'])
print('by family          :', catalog['by_family'])

In [ ]:
metadata_columns = {'subject_id', 'recording_id', 'epoch_id', 'onset_s', 'stage',
                    'ch_name', 'data_origin', 'split', 'sfreq', 'epoch_seconds',
                    'timestamp'}
feature_columns = [c for c in table.columns if c not in metadata_columns]
print(f'{len(feature_columns)} feature columns')
missing = table[feature_columns].isna().mean().sort_values(ascending=False)
print('columns with any missing values:')
print(missing[missing > 0].head(10))

`feature_delta_norm*` are NaN on the first epoch of each recording by design:
they are **non-causal** inter-epoch features and are excluded from every causal
ablation.